In [1]:
import torch
torch.__version__

'2.8.0+cpu'

In [2]:
import torch
torch.cuda.is_available()

False

In [3]:
import torch
print(torch.version.cuda)

None


In [4]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))


2.20.0
[]


In [5]:
!nvidia-smi

NVIDIA-SMI has failed because you do not have suffient permissions. Please try running as an administrator.


# STRATIFIED K-FOLD+EARLY STOP

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import StratifiedKFold
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import time
import os

In [ ]:
# EarlyStopping class (same as before)
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [ ]:
# Set device and define model and optimizer
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
# Example: Define a simple model (ViT or any other model can be used)
pretrained_vit_weights = torchvision.models.ViT_B_16_Weights.DEFAULT 
pretrained_vit = torchvision.models.vit_b_16(weights=pretrained_vit_weights).to(device)

In [ ]:
# Freeze base layers
for parameter in pretrained_vit.parameters():
    parameter.requires_grad = False

In [ ]:
# Change classifier head
class_names = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j']
pretrained_vit.heads = nn.Linear(in_features=768, out_features=len(class_names)).to(device)

In [ ]:
# Define dataset and transforms
train_dir = r'G:\islaphabets\NUS-Hand-Posture-Dataset-II\NUS Hand Posture dataset-II\Train Augment'
test_dir = r'G:\islaphabets\NUS-Hand-Posture-Dataset-II\NUS Hand Posture dataset-II\Test Augment'

In [ ]:
pretrained_vit_transforms = pretrained_vit_weights.transforms()

In [ ]:
# Create dataloaders (similar to previous)
def create_dataloaders(train_dir, transform, batch_size, num_workers):
    train_data = datasets.ImageFolder(train_dir, transform=transform)
    return DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True), train_data.classes

In [ ]:
# K-fold cross-validation setup
k_folds = 5
batch_size = 32
num_workers = os.cpu_count()

In [ ]:
# Create the k-fold Cross Validator
kf = StratifiedKFold(n_splits=k_folds, shuffle=True)

In [ ]:
# Train and validate model using k-fold cross-validation
def train_with_kfold_cross_validation(model, dataset, k_folds, optimizer, loss_fn, epochs, device, early_stopping):
    # Get class labels from the dataset
    labels = dataset.targets
    total_start_time = time.time()

    for fold, (train_idx, val_idx) in enumerate(kf.split(dataset.imgs, labels)):
        print(f"Fold {fold + 1}/{k_folds}")
        
        # Split the dataset into train and validation sets for this fold
        train_subset = Subset(dataset, train_idx)
        val_subset = Subset(dataset, val_idx)
        
        # Create dataloaders for this fold
        train_dataloader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
        val_dataloader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
        
        # Reinitialize model, optimizer, and early stopping for each fold
        model_copy = model.to(device)
        optimizer_copy = optimizer(model_copy.parameters(), lr=1e-3)
        early_stopping_copy = EarlyStopping(patience=5, delta=0.01)
        
        # Start training for this fold
        best_val_loss = float('inf')
        for epoch in range(epochs):
            start_time = time.time()
            
            model_copy.train()
            running_loss = 0.0
            correct_train = 0
            total_train = 0
            for images, labels in train_dataloader:
                images, labels = images.to(device), labels.to(device)
                
                optimizer_copy.zero_grad()
                outputs = model_copy(images)
                loss = loss_fn(outputs, labels)
                loss.backward()
                optimizer_copy.step()
                
                running_loss += loss.item()
                
                # Train accuracy
                _, predicted = torch.max(outputs, 1)
                total_train += labels.size(0)
                correct_train += (predicted == labels).sum().item()

            # Validation step
            model_copy.eval()
            val_loss = 0.0
            correct_val = 0
            total_val = 0
            with torch.no_grad():
                for images, labels in val_dataloader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model_copy(images)
                    loss = loss_fn(outputs, labels)
                    val_loss += loss.item()

                    # Validation accuracy
                    _, predicted = torch.max(outputs, 1)
                    total_val += labels.size(0)
                    correct_val += (predicted == labels).sum().item()

            avg_train_loss = running_loss / len(train_dataloader)
            avg_val_loss = val_loss / len(val_dataloader)
            train_accuracy = 100 * correct_train / total_train
            val_accuracy = 100 * correct_val / total_val
            
            epoch_time = time.time() - start_time
            
            print(f"Epoch [{epoch+1}/{epochs}], "
                  f"Train Loss: {avg_train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, "
                  f"Val Loss: {avg_val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%, "
                  f"Epoch Time: {epoch_time:.2f} seconds")

            # Early stopping check
            early_stopping_copy(avg_val_loss)
            if early_stopping_copy.early_stop:
                print(f"Early stopping triggered in fold {fold + 1}!")
                break

        print(f"Finished fold {fold + 1}.")
        
    total_time = time.time() - total_start_time
    print(f"Total Training Time for {k_folds}-fold Cross Validation: {total_time:.2f} seconds")

# Initialize the loss function and optimizer
optimizer = torch.optim.Adam
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
# Train with k-fold cross-validation
train_with_kfold_cross_validation(pretrained_vit,
                                  dataset=datasets.ImageFolder(train_dir, transform=pretrained_vit_transforms),
                                  k_folds=k_folds,
                                  optimizer=optimizer,
                                  loss_fn=loss_fn,
                                  epochs=10,
                                  device=device,
                                  early_stopping=EarlyStopping(patience=5, delta=0.01))